# Proactive Fraud Detection Model
### Dataset Handling No

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


In [ ]:
df = pd.read_csv("Fraud.csv")

In [ ]:
df.shape

The dataset contains 6,362,620 transaction records and 10 features.

In [ ]:
df.head()

Each row represents an individual financial transaction including transaction type, amount, and balance information.


In [ ]:
df.info()

The dataset includes numerical and categorical variables with no explicit missing values.

In [ ]:
df['isFraud'].value_counts()

The target variable `isFraud` is highly imbalanced, which is typical in real-world fraud detection problems and influences model evaluation.


In [ ]:
df['type'].value_counts()

Fraudulent transactions are more commonly observed in specific transaction types such as TRANSFER and CASH_OUT.

In [ ]:
df.isnull().sum()

No explicit missing values were found in the dataset. Zero balances observed for some destination accounts are structural and correspond to merchant transactions, not missing data.


In [ ]:
df.duplicated().sum()

No duplicate transaction records were found in the dataset.

In [ ]:
df['amount'].describe()

Transaction amounts are highly right-skewed with extreme values. These outliers were not removed, as fraudulent transactions naturally occur at extreme values. Instead, transformations and derived features were used.


In [ ]:
df['balanceDiffOrig'] = df['oldbalanceOrg'] - df['newbalanceOrig']
df['balanceDiffDest'] = df['newbalanceDest'] - df['oldbalanceDest']

Balance difference features were created to capture abnormal fund movements, which are strong indicators of fraudulent behavior.


In [ ]:
corr = df[['oldbalanceOrg', 'newbalanceOrig',
           'oldbalanceDest', 'newbalanceDest']].corr()
corr

Strong correlations were observed between balance variables before and after transactions. To reduce multicollinearity, balance difference features were preferred during modeling.


In [ ]:
df_model = df.drop(['nameOrig', 'nameDest'], axis=1)

Customer identifiers were removed as they do not provide predictive value and may introduce noise into the model.


In [ ]:
le = LabelEncoder()
df_model['type'] = le.fit_transform(df_model['type'])

The categorical variable `type` was encoded into numerical form using label encoding to make it suitable for machine learning algorithms.

In [ ]:
X = df_model.drop('isFraud', axis=1)
y = df_model['isFraud']

The dataset was split into features (X) and target variable (y), where `isFraud` represents fraudulent transactions.

### Selected Features
- Transaction type
- Transaction amount
- Sender and receiver balances
- Balance difference features
- Flagged fraud indicator

These variables were selected based on domain knowledge and their relevance to fraudulent behavior.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

The dataset was split into training (70%) and validation (30%) sets. Stratified sampling was used to preserve the class imbalance in both datasets.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

Class imbalance was addressed by assigning higher weights to the minority (fraud) class, ensuring the model focuses on detecting fraudulent transactions.

In [ ]:
rf_model.fit(X_train, y_train)

A Random Forest classifier was trained on the calibration dataset to learn patterns associated with fraudulent transactions.

### Model Selection Rationale
A Random Forest classifier was chosen because it:
- Handles large datasets efficiently
- Captures non-linear relationships
- Is robust to noise and outliers
- Provides feature importance for interpretability

These properties make it suitable for fraud detection problems.


In [ ]:
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]


Predictions were generated on the validation dataset to evaluate the model’s ability to detect fraudulent transactions.


In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

The confusion matrix highlights the trade-off between detecting fraudulent transactions and minimizing false alarms.

In [ ]:
print(classification_report(y_test, y_pred))

Precision, recall, and F1-score were used to evaluate model performance. Recall was prioritized to minimize missed fraudulent transactions.

In [ ]:
roc_auc = roc_auc_score(y_test, y_prob)
roc_auc


The ROC–AUC score indicates strong discriminative performance, demonstrating the model’s effectiveness in distinguishing fraudulent and legitimate transactions.


Accuracy was not used as the primary evaluation metric due to the highly imbalanced nature of the dataset. Metrics such as recall and ROC–AUC provide a more reliable assessment of fraud detection performance.


### Key Factors Predicting Fraudulent Transactions
The most important factors contributing to fraudulent transactions are:
- Transaction type, particularly TRANSFER and CASH_OUT
- High transaction amounts
- Sudden depletion of sender account balance
- Destination accounts with zero or very low initial balance
- Large discrepancies between transaction amount and balance changes
- Transactions flagged due to high transfer amounts


Yes, these factors align with real-world fraud behavior. Fraudsters typically aim to quickly empty compromised accounts by transferring funds to mule accounts or cashing out large amounts. Legitimate users rarely exhibit such abrupt balance changes or transfer funds to inactive accounts.


### Recommended Fraud Prevention Strategies
- Real-time transaction risk scoring
- Dynamic transaction limits for high-risk users
- Multi-factor authentication for large or unusual transactions
- Velocity checks to detect rapid consecutive transactions
- Temporary account freezes for suspicious activity
- Continuous monitoring and model retraining


### Measuring the Effectiveness of Fraud Prevention
The effectiveness of implemented prevention measures can be evaluated using:
- Reduction in fraud rate and financial loss
- Monitoring false positive and false negative rates
- Customer complaint and transaction approval trends
- A/B testing comparing fraud metrics before and after deployment
